In [1]:
# If you don't have the moduel please install them for this notebook
#!pip install Bio

In [2]:
%pip install numpy pandas biopython matplotlib seaborn scikit-learn



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import re
import numpy as np
import pandas as pd
from collections import Counter
import itertools
from Bio.SeqUtils.ProtParam import ProteinAnalysis

from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

# Read in data that was generated from the analysis and preprocessing steps

In [4]:
# Read in data
data = pd.read_csv('metadata_org_w_features.csv')

# Information and sample data

In [5]:
# information on data
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 681 entries, 0 to 680
Columns: 589 entries, Entry to Gravy
dtypes: float64(584), int64(1), str(4)
memory usage: 3.1 MB


In [6]:
# describe on data
data.describe()

,SequenceLength,A,C,D,E,F,G,H,I,K,...,DDD,MolecularWeight,IsoelectricPoint,Aromaticity,InstabilityIndex,Flexibility,Helix,Sheet,Turn,Gravy
count,681.000000,681.000000,681.000000,681.000000,681.000000,681.000000,681.000000,681.000000,681.000000,681.000000,...,681.000000,681.000000,681.000000,681.000000,681.000000,681.000000,681.000000,681.000000,681.000000,681.000000
mean,888.660793,0.071173,0.023348,0.044993,0.055845,0.046743,0.065586,0.022028,0.056255,0.047262,...,0.002206,99103.867857,7.097584,0.096863,42.160570,0.995021,0.304995,0.279123,0.389817,-0.051027
std,827.859592,0.022832,0.012300,0.013231,0.017925,0.015308,0.018840,0.007750,0.018204,0.017596,...,0.003108,92308.456460,1.551349,0.021673,7.412366,0.006859,0.029081,0.032611,0.043874,0.325872
min,103.000000,0.024283,0.000000,0.007605,0.005714,0.007782,0.016807,0.004065,0.009840,0.002915,...,0.000000,11710.270000,4.383673,0.033074,15.359896,0.974438,0.222539,0.183150,0.248630,-0.757339
25%,384.000000,0.056695,0.015251,0.037879,0.044280,0.035479,0.052764,0.016825,0.044171,0.034237,...,0.000000,42771.083500,5.723255,0.082587,37.259593,0.991538,0.284024,0.257261,0.361333,-0.284269
50%,676.000000,0.067460,0.021417,0.046046,0.058594,0.044747,0.065963,0.021544,0.056726,0.048017,...,0.001377,74697.663500,6.556855,0.094744,42.218182,0.996686,0.302326,0.277000,0.382857,-0.120350
75%,1015.000000,0.081784,0.030753,0.053254,0.067669,0.055838,0.078212,0.026769,0.068000,0.058943,...,0.003229,112207.097800,8.660054,0.109974,46.958534,0.999503,0.323615,0.300946,0.416667,0.136484
max,7096.000000,0.164000,0.074873,0.116598,0.123102,0.110701,0.153509,0.056093,0.125461,0.104956,...,0.031637,794048.925800,11.114366,0.180812,66.024733,1.013136,0.403005,0.419863,0.546125,1.008772


In [7]:
data.head()

,Entry,CleanSequence,Selected_PDB,ProteinClass,SequenceLength,A,C,D,E,F,...,DDD,MolecularWeight,IsoelectricPoint,Aromaticity,InstabilityIndex,Flexibility,Helix,Sheet,Turn,Gravy
0,P21611,MGKAAAVVLVTLVALLGLAQADLTPKVQVYSRFPASAGTKNVLNCF...,3p73,MHC,119,0.109244,0.016807,0.058824,0.042017,0.058824,...,0.000000,13041.7928,5.834204,0.109244,22.404202,0.996347,0.336134,0.277311,0.369748,-0.047059
1,Q66GT5,MAASAWLEAGLARVLFYPTLLYTVFRGRVRGPAHRDWYHRIDHTVL...,3rgo,Phosphatase,193,0.108808,0.015544,0.020725,0.067358,0.025907,...,0.000000,21942.2575,9.739448,0.082902,37.915078,0.992205,0.367876,0.186528,0.383420,-0.161140
2,Q9Y006,MNLTIKEEDFTNTFMKNEESFNTFRVTKVKRWNAKRLFKILFVTVF...,3qvc,Protease,451,0.028825,0.008869,0.048780,0.068736,0.079823,...,0.002227,51692.5641,8.043414,0.137472,37.618204,1.002078,0.305987,0.290466,0.434590,-0.249667
3,P05622,MGLPGVIPALVLRGQLLLSVLWLLGPQTSRGLVITPPGPEFVLNIS...,1aya,RTK,1098,0.043716,0.017304,0.056466,0.069217,0.030055,...,0.003650,122788.2434,4.994010,0.083789,47.564763,0.999045,0.281421,0.311475,0.385246,-0.203097
4,P06343,MALQIPSLLLLAAVVVLTVLSSPGTEGGNSERHFVHQFQPFCYFTN...,1d9k,MHC,263,0.041825,0.019011,0.022814,0.076046,0.038023,...,0.000000,29966.8989,8.991680,0.091255,46.750989,0.995690,0.262357,0.254753,0.399240,-0.303042


In [8]:
# class imbalance check - I added :) 

# Count the number of proteins in each class
class_counts = data["ProteinClass"].value_counts()

# Create a summary table
summary = pd.DataFrame({
    "Count": class_counts,
    "Percentage": data["ProteinClass"].value_counts(normalize=True) * 100
})

print(summary)
# obviosuly minority class is much less than the majority class. We will need to address this imbalance in our model training.

              Count  Percentage
ProteinClass                   
GPCR            144   21.145374
Channel         142   20.851689
Protease         79   11.600587
ATPase           66    9.691630
RTK              64    9.397944
Phosphatase      56    8.223201
Ser:Thr          49    7.195301
MHC              45    6.607930
Integrin         21    3.083700
Aquaporin        15    2.202643


# Extract the relevant fields from the dataset

In [9]:
# Entry and ProteinClass
df = data.loc[:,['Entry', 'ProteinClass']]
# PDB
selected_PDB = data.loc[:,['Selected_PDB']]
# Sequence
seq = data.loc[:,['CleanSequence']]
# Sequence Length
seq_L = data.loc[:,['SequenceLength']]
# Amino Acid Frequencies
freq = data.iloc[:, 5 : 5+20]
# All possible dipeptide frequencies
dipep = data.iloc[:, 5+20 : 5+20 + 400]
# Reduced Amino Acid Alphabet Frequencies
red_freq = data.iloc[:, 5+20 + 400 : 5+20 + 400 + 5]
# N-Gram Profiles of Reduced Amino Acid Alphabet
red_ngram = data.iloc[:, 5+20 + 400 + 5 : 5+20 + 400 + 5 + 150]
# Protein Properties
prop = data.iloc[:, 5+20 + 400 + 5 + 150 :]

In [10]:
prop.head()

,MolecularWeight,IsoelectricPoint,Aromaticity,InstabilityIndex,Flexibility,Helix,Sheet,Turn,Gravy
0,13041.7928,5.834204,0.109244,22.404202,0.996347,0.336134,0.277311,0.369748,-0.047059
1,21942.2575,9.739448,0.082902,37.915078,0.992205,0.367876,0.186528,0.383420,-0.161140
2,51692.5641,8.043414,0.137472,37.618204,1.002078,0.305987,0.290466,0.434590,-0.249667
3,122788.2434,4.994010,0.083789,47.564763,0.999045,0.281421,0.311475,0.385246,-0.203097
4,29966.8989,8.991680,0.091255,46.750989,0.995690,0.262357,0.254753,0.399240,-0.303042


# You can use this section if you wanted to add more fetures to the data

In [11]:
## To Do ## 
# Add one new feature to this data, you don't need to use it for the model
def charge_at_pH(seq, pH = 7.0):
        """Calculate the charge of a protein at given pH."""
        analysis = ProteinAnalysis(str(seq))
        return analysis.charge_at_pH(pH)

# Calculate the feature for each protein
data["charge_pH7"] = data["CleanSequence"].apply(charge_at_pH)

/var/folders/90/6rghjrw16jq9p_g4gh3cvs840000gn/T/ipykernel_32931/2852606358.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data["charge_pH7"] = data["CleanSequence"].apply(charge_at_pH)


# Construct Dataset with any of the extracted relevant fields

In [12]:
# This is an example of construction of the data with freq and red_freq, but you can add and make any field
Dataset = pd.concat([df,freq, prop,red_freq, red_ngram], axis=1)
Dataset.head()
# detect missing values 
missing = Dataset.isna().sum()
print(missing[missing > 0])

Series([], dtype: int64)


In [13]:
## To Do ##

Dataset = pd.concat([df,freq, prop, red_ngram], axis=1)
Dataset.head()

,Entry,ProteinClass,A,C,D,E,F,G,H,I,...,DHG,DHF,DHS,DHH,DHD,DDG,DDF,DDS,DDH,DDD
0,P21611,MHC,0.109244,0.016807,0.058824,0.042017,0.058824,0.058824,0.025210,0.016807,...,0.000000,0.000000,0.000000,0.000000,0.008547,0.000000,0.008547,0.000000,0.000000,0.000000
1,Q66GT5,Phosphatase,0.108808,0.015544,0.020725,0.067358,0.025907,0.051813,0.041451,0.046632,...,0.000000,0.000000,0.005236,0.000000,0.000000,0.005236,0.005236,0.005236,0.000000,0.000000
2,Q9Y006,Protease,0.028825,0.008869,0.048780,0.068736,0.079823,0.053215,0.015521,0.062084,...,0.004454,0.002227,0.002227,0.002227,0.002227,0.000000,0.002227,0.004454,0.002227,0.002227
3,P05622,RTK,0.043716,0.017304,0.056466,0.069217,0.030055,0.061020,0.021858,0.050091,...,0.003650,0.000912,0.003650,0.001825,0.000912,0.004562,0.000912,0.003650,0.002737,0.003650
4,P06343,MHC,0.041825,0.019011,0.022814,0.076046,0.038023,0.072243,0.030418,0.041825,...,0.003831,0.000000,0.007663,0.003831,0.000000,0.000000,0.003831,0.003831,0.000000,0.000000


# Processing Steps

In [14]:
## To Do ##

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder


def processing_data(df):
    # Define X and y
    y = df['ProteinClass']
    X = df.drop(['Entry', 'ProteinClass'], axis=1)
    if 0:
        print("Feature sample:")
        print(X.head())
        print("\nTarget sample:")
        print(y.head())
        # Entry is removed because it is just an identifier.
        # ProteinClass is removed because giving the model the answer as an input would be cheating.
        # axis=1 means remove columns.

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)

    # Feature scaling (important for logistic regression, gradient boosting, MLP, etc.)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
   

    ## To Do ##
    # Add a max_min normalization for the X_train and X_test below
    minmax_scaler = MinMaxScaler()
    X_train_minmax = minmax_scaler.fit_transform(X_train)
    X_test_minmax = minmax_scaler.transform(X_test)

    # Defined labels
    labels = np.array(['ATPase', 'Aquaporin', 'Channel', 'GPCR', 'Integrin', 'MHC',
       'Phosphatase', 'Protease', 'RTK', 'Ser:Thr'])
    # Create a Label Encoder
    label_encoder = LabelEncoder()
    # Fit the label encoder on all possible labels, will make the labels [0, 1, 2, 3 ...]
    label_encoder.fit(labels)
    # Transform both training and testing labels
    y_train_label_encoded = label_encoder.transform(y_train)
    y_test_label_encoded = label_encoder.transform(y_test)
    print('first sample label encoded:', y_train_label_encoded[0])

    ## To Do ##
    # Add a one hot encoder to the labels
    # Create a One Hot Encoder
    encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

    # Fit the one hot encoder on all possible labels, will make the labels 0 -> [1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    encoder.fit(df[["ProteinClass"]])

    # Transform both training and testing labels like the above code
    y_train_onehot = encoder.transform(y_train.to_frame())
    y_test_onehot = encoder.transform(y_test.to_frame())

    ### This is where you can start to add more normalization or processing approches if you would like ###

    return label_encoder, X_train, X_train_scaled, X_test, X_test_scaled, X_train_minmax, X_test_minmax, y_train, y_train_label_encoded, y_test, y_test_label_encoded


In [15]:
label_encoder, X_train, X_train_scaled, X_test, X_test_scaled, X_train_minmax, X_test_minmax, y_train, y_train_label_encoded, y_test, y_test_label_encoded= processing_data(Dataset)

first sample label encoded: 3


# Creating Model

In [16]:
# random model
class random_model:
    def __init__(self):
        pass
    def fit(self,X, y):
        pass
    def predict(self,X):
        pred = []
        for x in X:
            pred.append(np.random.randint(10))
        return np.array(pred)
    #random model is 1/10 accuracy 
# Initialize my random model
my_model = random_model()
# Initialize Logistic Regression model
lr_model = LogisticRegression( C = 1)

## To Do ##
# Create one model here like random forest, leanear model, etc and initialize it
# you can use sklear, torch, etc 
from sklearn.ensemble import RandomForestClassifier

# Create the model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)


# Train Model

In [17]:
# Fit My Model to the training data
my_model.fit(X_train, y_train)

# Fit the model to the training data
lr_model.fit(X_train_scaled, y_train)

## To Do ##
# Fit your model to the training data you can use
# raw, scaled, or other forms
rf_model.fit(X_train, y_train)
#encoding is important during evaluation, not training.


,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [18]:
def evaluate_model(model, X_test, y_test, model_name):

    y_pred = model.predict(X_test)

    # Evaluation
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\n=== {model_name} ===")
    print(f"Accuracy for testing data: {accuracy:.4f}")
    return y_pred

In [19]:
# Evaluate my model
y_pred_my_model = evaluate_model(my_model, X_test_scaled, y_test_label_encoded, model_name="My Random Model")
# Convert back to labels
y_pred_my_model = label_encoder.inverse_transform(y_pred_my_model)
# Print my model prediction
# print(y_pred_my_model)
#print(y_pred_my_model[:10])
# Evaluate Logistic Regression
y_pred_lr_model = evaluate_model(lr_model, X_test_scaled, y_test, model_name="Logistic Regression")
# Print lr model prediction
# print(y_pred_my_model)
# print(y_pred_lr_model[:10])
## To Do ##
# Evaluate your model
y_pred_rf_model = evaluate_model(rf_model, X_test, y_test, model_name="Random Forest")
#print(y_pred_rf_model[:10])


=== My Random Model ===
Accuracy for testing data: 0.0876

=== Logistic Regression ===
Accuracy for testing data: 0.7299

=== Random Forest ===
Accuracy for testing data: 0.7372


# Save Format

In [20]:
def save_predictions(_fn, _y_pred, _df):
    import csv
    with open(_fn, 'w') as fout:
        writer = csv.writer(fout, delimiter=',', lineterminator='\n')
        writer.writerow(['Entry', 'ProteinClass'])
        for y, Entry in zip(_df['Entry'], _y_pred):
            writer.writerow([y, Entry])